In [1]:
import pandas as pd
import json
import datasets
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

/home/samoed/Desktop/dialogmteb/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from datasets import Dataset, DatasetDict, Features, Value, List

def process_ds(data: list)->Dataset:
    all_keys = set()
    for row in data:
        all_keys.update(row.keys())

    # Convert into a dataset, filling missing with None
    normalized_data = [
        {k: row.get(k, None) for k in all_keys}
        for row in data
    ]

    part_ds = Dataset.from_list(normalized_data)
    return part_ds

def process_ds_dict(data: dict[str, dict]) -> DatasetDict:
    all_keys = set()
    for ds in data.values():
        for row in ds:
            all_keys.update(row.keys())

    # features = {
    #     col: Value("string")
    #     for col in all_keys
    # }
    # features["dialog"] = List({"content": Value("string"), "role": Value("string")})
    # features = Features(features)

    # Convert into a dataset, filling missing with None
    for name, ds in data.items():
        ds = [
            {k: row.get(k, "none") for k in all_keys}
            for row in ds
        ]
        data[name] = Dataset.from_list(ds)
    return DatasetDict(data)

In [3]:
path = Path("/home/samoed/Downloads/dstc8-schema-guided-dialogue/")

In [4]:
dialogues_files = list(path.glob("./*/dialogues_*.json"))

In [ ]:
dialogues = defaultdict(lambda: defaultdict(list))

for dialogue_files in dialogues_files:

    split = dialogue_files.parent.name

    with dialogue_files.open() as f:
        data = json.load(f)

    for dialogue in tqdm(data, desc=str(dialogue_files)):
        dialogue_id = dialogue["dialogue_id"]
        services = dialogue["services"]
        cur_dialog = []
        for replic in dialogue["turns"]:
            cur_dialog.append(
                {
                    "role": replic["speaker"].lower(),
                    "content": replic["utterance"],
                }
            )

            for frame in replic["frames"]:
                cur_subset = frame["service"].split('_')[0]
                actions = frame["actions"]
                cur_actions = {}
                for action in actions:
                    act = action["act"].lower()
                    if len(action["canonical_values"]) == 0:
                        continue
                    canonical_value = action["canonical_values"][0]
                    value = action["values"][0]
                    slot = action["slot"]
                    cur_actions[f'{act}_{slot}_canonical'] = canonical_value
                    cur_actions[f'{act}_{slot}_value'] = value
                dialogues[cur_subset][split].append(
                    {
                        'dialogue_id': dialogue_id,
                        'services': services,
                        "dialog": cur_dialog,
                        **cur_actions,
                    }
                )
                


/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_001.json: 100%|██████████| 128/128 [00:00<00:00, 3839.43it/s]


/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_002.json: 100%|██████████| 128/128 [00:00<00:00, 8587.19it/s]
/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_003.json: 100%|██████████| 128/128 [00:00<00:00, 4473.92it/s]
/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_004.json: 100%|██████████| 128/128 [00:00<00:00, 8188.13it/s]
/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_005.json: 100%|██████████| 128/128 [00:00<00:00, 953.44it/s]
/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_006.json: 100%|██████████| 128/128 [00:00<00:00, 9609.80it/s]
/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_007.json: 100%|██████████| 68/68 [00:00<00:00, 15907.01it/s]
/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_008.json: 100%|██████████| 128/128 [00:00<00:00, 3388.93it/s]
/home/samoed/Downloads/dstc8-schema-guided-dialogue/dev/dialogues_009.json: 100%|██████████| 128/128 [00:

In [7]:
dialogues["Restaurants"].keys()

dict_keys(['dev', 'test', 'train'])

In [8]:
dialogues["Restaurants"]["dev"][0]

{'dialogue_id': '1_00000',
 'services': ['Restaurants_2'],
 'dialog': [{'role': 'user',
   'text': 'I want to make a restaurant reservation for 2 people at half past 11 in the morning.'},
  {'role': 'system',
   'text': 'What city do you want to dine in? Do you have a preferred restaurant?'},
  {'role': 'user',
   'text': 'Please find restaurants in San Jose. Can you try Sino?'},
  {'role': 'system',
   'text': 'Confirming: I will reserve a table for 2 people at Sino in San Jose. The reservation time is 11:30 am today.'},
  {'role': 'user', 'text': "Yes, thanks. What's their phone number?"},
  {'role': 'system',
   'text': 'Your reservation has been made. Their phone number is 408-247-8880.'},
  {'role': 'user',
   'text': "What's their address? Do they have vegetarian options on their menu?"},
  {'role': 'system',
   'text': 'The street address is 377 Santana Row #1000. They have good vegetarian options.'},
  {'role': 'user', 'text': 'Thanks very much.'},
  {'role': 'system', 'text': 

In [12]:
for k, v in dialogues.items():
    ds = process_ds_dict(v)
    ds.push_to_hub("DeepPavlov/DSTC8", k)

Creating parquet from Arrow format: 100%|██████████| 6/6 [00:00<00:00, 114.30ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████|  472kB /  472kB, 87.3kB/s  

Processing Files (1 / 1)                : 100%|██████████|  472kB /  472kB, 84.2kB/s  
New Data Upload                         : 100%|██████████|  472kB /  472kB, 84.2kB/s  
                                        : 100%|██████████|  472kB /  472kB            
Creating parquet from Arrow format: 100%|██████████| 7/7 [00:00<00:00, 148.84ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (1 / 1)                : 100%|██████████|  628kB /  628kB,  262kB/s  

Processing Files (1 / 1)                : 100%|██████████|  628kB /  628kB,  242kB/s  
New Data Upload                         : 100%|██████████|  628kB /  628kB,  242kB/s  
                                        : 100%|████████

In [11]:
ds["dev"]["dialog"]

Column([[{'role': 'user', 'text': 'I want to make a restaurant reservation for 2 people at half past 11 in the morning.'}, {'role': 'system', 'text': 'What city do you want to dine in? Do you have a preferred restaurant?'}, {'role': 'user', 'text': 'Please find restaurants in San Jose. Can you try Sino?'}, {'role': 'system', 'text': 'Confirming: I will reserve a table for 2 people at Sino in San Jose. The reservation time is 11:30 am today.'}, {'role': 'user', 'text': "Yes, thanks. What's their phone number?"}, {'role': 'system', 'text': 'Your reservation has been made. Their phone number is 408-247-8880.'}, {'role': 'user', 'text': "What's their address? Do they have vegetarian options on their menu?"}, {'role': 'system', 'text': 'The street address is 377 Santana Row #1000. They have good vegetarian options.'}, {'role': 'user', 'text': 'Thanks very much.'}, {'role': 'system', 'text': 'Is there anything else I can help you with?'}, {'role': 'user', 'text': "No, that's all. Thanks."}, 